# 00 · Protocol overview and how to use these notebooks

**Project:** Multimodal MRI fusion for psychosis classification via multiple kernel learning (MKL)
**Dataset:** COBRE
**Protocol:** `protocol/protocol.md` (draft v1.0, to be locked before any results)

These notebooks *are* the analysis. Each one does three things:
1. **explains** the concepts from the research protocol in markdown;
2. **implements** them in code cells;
3. **demonstrates** them on **synthetic data**, so everything runs before the COBRE data are extracted.

When the real data are in place, set `USE_REAL_DATA = True` in the notebook you are running.

| Notebook | Protocol sections | What it covers |
|---|---|---|
| 00 overview | §0–1, §9, §13–14 | objective, hypotheses, estimand, preregistration, reproducibility |
| 01 dataset audit | §2, §8 | COBRE, BIDS, single site, sample definition, exclusions, head motion, missing data |
| 02 structural features | §3.1–3.2 | T1w MRI, FreeSurfer, Desikan–Killiany atlas, thickness/area/volume, eTIV, Euler number |
| 03 functional features | §3.3–3.4 | BOLD, fMRIPrep, denoising, scrubbing, atlas, Pearson r, Fisher z, upper triangle, p ≫ n |
| 04 kernels and models | §4–5.2 | SVM, C, kernels, normalization, centering, early fusion, MKL, alignment, stacking |
| 05 nested cross-validation | §5.3–5.6 | overfitting, leakage, nested CV, stratification, repeats, seeds, pairing, thresholds |
| 06 metrics and inference | §6–7.4 | ROC-AUC, PR-AUC, balanced accuracy, ΔAUC, corrected CI, permutation tests, Holm, decision rule |
| 07 stability and confounds | §7.5, §8–10 | prediction/feature stability, confounds, sensitivity analyses, limitations |
| 08 visual exploration | results | figures of the final results and how to read them |

### How the notebooks share code
- Code cells that *define* functions carry the cell tag **`library`**. You can see tags in Jupyter via *View → Cell Toolbar → Tags*, or in VS Code in the cell's "…" menu.
- A later notebook loads an earlier one with `use_notebook("04_kernels_and_models.ipynb")`. This runs only the tagged definition cells, not the demos.
- Always run a notebook top to bottom (*Kernel → Restart & Run All*).

## 1. Objective

> Classify participants with a schizophrenia-spectrum diagnosis versus healthy controls from **structural** and **resting-state functional** MRI, and estimate whether fusing the two modalities with **multiple kernel learning (MKL)** improves on single-modality models.

### Key terms
- **Binary classification.** Learning a rule that maps a participant's measurements $x$ to a label $y \in \{0 = \text{control},\ 1 = \text{patient}\}$ from labelled examples, then applying that rule to *new* participants.
- **Modality.** One kind of measurement:
  - *Structural MRI* is brain anatomy from a T1-weighted scan, summarised as regional cortical thickness, surface area and subcortical volume (notebook 02).
  - *Resting-state functional MRI* is spontaneous brain activity over about 5 minutes, summarised as functional connectivity between brain regions (notebook 03).
- **Feature matrix.** Rows are participants and columns are features. $X_s$ holds the structural features and $X_f$ the functional features.
- **Multimodal fusion.** Combining modalities in one classifier. Fusion can happen at three points (notebook 04):
  - *early*: concatenating the features;
  - *in the kernel*: combining similarity matrices;
  - *late*: combining the models' outputs.
- **Kernel.** A matrix of pairwise similarities between participants. Each modality gets its own kernel, $K_s$ and $K_f$.
- **Multiple kernel learning (MKL).** A kernel method that *learns* how much weight each modality's kernel gets: $K = \beta_s K_s + \beta_f K_f$.

## 2. Hypotheses and the estimand

### Primary hypothesis (H1)
MKL provides *incremental* discrimination over the prespecified functional-connectivity model.

- **Incremental value.** The improvement from *adding* a modality to one you already have. The question is not "is MKL good?" but "is MKL better than functional connectivity alone?".
- **Prespecified baseline.** The comparator (the functional model) is fixed *before* any results are seen. Picking "the best unimodal model" after seeing results is a data-dependent choice that inflates apparent gains.
- **Estimand.** The exact quantity the study sets out to estimate:

$$\overline{\Delta\mathrm{AUC}} \;=\; \frac{1}{J}\sum_{j=1}^{J}\Big(\mathrm{AUC}^{(j)}_{\mathrm{MKL}} - \mathrm{AUC}^{(j)}_{\mathrm{functional}}\Big),\qquad J = 10\ \text{repeats} \times 5\ \text{folds} = 50$$

- **Paired outer-fold predictions.** Both models are scored on *the same* test participants in every fold. The difference therefore cancels out fold-to-fold variation in how easy the test participants happen to be.
- **Superiority** is declared only if all three conditions hold (notebook 06):
  1. the difference is positive;
  2. it is distinguishable from an appropriate null;
  3. it is at least as large as a prespecified **minimum meaningful difference**, $\delta = 0.02$ AUC.

### Secondary hypothesis (H2)
MKL outperforms naive fusion: early concatenation, equal-weight kernel averaging, and stacking.

Whether MKL actually down-weights a less informative modality is *described* using the learned $\beta$ values. It is not asserted as the reason for any difference.

## 3. Why the protocol is locked before results

- **Preregistration / analysis protocol.** Every analytic choice is written down before anyone sees outcome data.
- **Researcher degrees of freedom** (the "garden of forking paths"). Atlas, preprocessing, model, threshold, metric: every choice made *after* seeing results can nudge the answer toward significance, even with no intent to mislead.
- **Deviations log** (protocol §13). Any change after the lock is recorded with its reason, and with whether it was made before or after seeing outcome data.
- **Null-result interpretation, decided in advance.** If MKL fails to beat the functional model, this is evidence that fusion gave *no detectable incremental value under this dataset and design*. It is not a failed experiment. Fixing this reading in advance stops the interpretation from shifting after a negative result.
- **Lock record** (protocol §14). The lock date, SHA-256 hashes of the protocol and config files, and the git commit. The last cell of this notebook computes the hashes.

In [1]:
from pathlib import Path
import json

import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NOTEBOOKS = ROOT / "notebooks"


def use_notebook(name):
    """Run the definition cells (tagged "library") of another notebook in this namespace."""
    notebook = json.loads((NOTEBOOKS / name).read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code" and "library" in cell["metadata"].get("tags", []):
            exec("".join(cell["source"]), globals())


def load_yaml(path):
    with open(path, encoding="utf-8") as fh:
        return yaml.safe_load(fh)

## 4. Reproducibility: software versions
Results can change between library versions, for example when a solver's default changes. The protocol pins every version (`requirements.txt`, `configs/model_config.yaml`). This cell compares what is installed with what is pinned.

In [2]:
from importlib import metadata

pinned = load_yaml(ROOT / "configs" / "model_config.yaml")["software"]
for package in ["numpy", "scipy", "pandas", "scikit-learn", "nilearn", "nibabel", "joblib"]:
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = "not installed"
    flag = "" if installed == pinned[package] else "   <-- differs from the pinned version"
    print(f"{package:14s} pinned {pinned[package]:10s} installed {installed}{flag}")

numpy          pinned 2.5.1      installed 2.5.1
scipy          pinned 1.18.0     installed 1.18.0
pandas         pinned 3.0.5      installed 3.0.5
scikit-learn   pinned 1.8.0      installed 1.8.0
nilearn        pinned 0.14.1     installed 0.14.1
nibabel        pinned 5.4.2      installed 5.4.2
joblib         pinned 1.5.3      installed 1.5.3


## 5. The prespecified configuration
Every analytic choice lives in `configs/`. The notebooks read these files instead of hard-coding values, so the locked config is the single source of truth. What each setting means is explained in notebooks 04–06.

In [3]:
model_cfg = load_yaml(ROOT / "configs" / "model_config.yaml")
cv_cfg = load_yaml(ROOT / "configs" / "cv_config.yaml")
print("SVM C grid:           ", model_cfg["svm"]["C_grid"])
print("MKL beta grid:        ", model_cfg["mkl"]["beta_grid"])
print("Kernel:               ", model_cfg["kernel"])
print("Threshold rule:       ", model_cfg["threshold"]["method"])
print("Outer CV:             ", cv_cfg["outer"])
print("Inner CV:             ", cv_cfg["inner"])
print("Permutations:         ", cv_cfg["permutation"]["n_permutations"])
print("Minimum meaningful ΔAUC (δ):", cv_cfg["inference"]["min_meaningful_delta_auc"])
print("\nModels:")
for m in model_cfg["models"]:
    print(f"  {m['name']:24s} role = {m['role']}")

SVM C grid:            [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
MKL beta grid:         [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
Kernel:                {'feature_scaling': 'zscore', 'type': 'linear', 'normalization': 'trace', 'centering': False}
Threshold rule:        youden_inner_oof
Outer CV:              {'scheme': 'repeated_stratified_kfold', 'n_splits': 5, 'n_repeats': 10}
Inner CV:              {'scheme': 'stratified_kfold', 'n_splits': 5}
Permutations:          1000
Minimum meaningful ΔAUC (δ): 0.02

Models:
  structural               role = baseline
  functional               role = primary_comparator
  early_fusion             role = secondary
  equal_weight             role = secondary
  mkl                      role = primary
  stacking                 role = secondary
  confounds_only           role = confound_baseline
  mkl_alignment            role = sensitivity
  functional_residualized  role = sensitivity
  mkl_residualized         role = sensitivity


## 6. Lock record: file fingerprints
A **SHA-256 hash** is a fingerprint of a file's exact bytes: change one character and the hash changes completely. Recording the hashes at lock time lets anyone check later that the protocol and configs were not edited. Copy these values into protocol §14 when you lock.

In [4]:
import hashlib

for path in [ROOT / "protocol" / "protocol.md", *sorted((ROOT / "configs").glob("*.yaml")), ROOT / "configs" / "seeds.txt"]:
    print(hashlib.sha256(path.read_bytes()).hexdigest(), path.relative_to(ROOT))

7d7fcf3b25e7c1aee1b153d98694a4f29f0546df9fc07962091c6958f479bd44 protocol\protocol.md
7135e4a40c14e379d70d91069c85fcd49559fb047e29b2a645287ad6e613bc10 configs\cv_config.yaml
5bfd3620059a8fd937e9d871dda60c8a7652141efc13165c1a54c681f1a71deb configs\data_config.yaml
b2e83d8958ccde09ea8c6c58896d5f3a0c48313f18b6f16589cd4fae79528f40 configs\model_config.yaml
36a941569b163d36b98e17f7c4a87da56c85e5f16d2ffb6fdb7f4f2bd45d1593 configs\seeds.txt


## Glossary: where each concept is explained

| Concept | Notebook |
|---|---|
| COBRE, BIDS, single site, transportability, sample-definition rule, exclusion flow, missing data, class imbalance | 01 |
| Framewise displacement (FD), DVARS, censoring (scrubbing) | 01, 03 |
| T1w MRI, FreeSurfer, Desikan–Killiany atlas, cortical thickness, surface area, subcortical volume, eTIV, Euler number | 02 |
| BOLD, fMRIPrep, MNI space, confound regression, global signal regression, Schaefer atlas, Pearson r, Fisher z, upper triangle, tSNR, p ≫ n | 03 |
| SVM, hinge loss, L2 regularization, C, kernel, Gram matrix, PSD | 04 |
| Kernel normalization, centering, early/kernel/late fusion, MKL, simplex constraint, kernel alignment, stacking, out-of-fold predictions | 04 |
| Overfitting, data leakage, nested CV, stratification, repeated CV, seeds, paired splits, decision threshold, Youden's J | 05 |
| Confusion matrix, sensitivity, specificity, PPV, balanced accuracy, ROC-AUC, PR-AUC | 06 |
| Paired ΔAUC, Nadeau–Bengio correction, permutation test, exchangeability, plus-one p-value, Holm correction, δ, decision rule | 06 |
| Prediction stability, feature stability, weight similarity, sign consistency, network aggregation | 07 |
| Confounds, confound-only baseline, cross-validated confound regression, sensitivity analyses, limitations | 07 |